# Simulation Code T = 2

## Import libraries

In [2]:
#import torch
import pandas as pd
import time
from utils.dynamicRieszFunctions import *

## Estimation Settings

In [3]:
lasso_cv_settings = {
    'b_degree' : 1,
    'cv_folds' : 5,
    'random_state' : 42
}

lasso_a_settings = {
    'lambda_val' : 0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' : 0.1, # CHANGED FROM "CV"
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

lasso_f_settings = {
    'lambda_val' : 0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' :  0.1, # CHANGED FROM "CV"
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

rf_a_settings = {
    'poly_degree' : 1,
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' : -1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}

rf_f_settings = {
    'poly_degree' : 1,
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' : -1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}

net_a_settings = {
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}

net_f_settings = {
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}


## Generate data 

#### Define treatment probability function

In [4]:
# Bounds (only for truncated distributions)
lower = 0.3
upper = 0.7

# Define logistic function
def logistic(x):
    return torch.exp(x) / (1 + torch.exp(x))

# Define a truncated logistic function
def truncated_logistic(x):
    return lower + (upper - lower) * logistic(x)

# Define func_link function (nonlinear probability function from Bradic)
def func_link(x):
    return (x > 0) * torch.abs(x) / (torch.abs(x) + 1) + (x < 0) / (torch.abs(x) + 1)

# Define a truncated func_link function
def truncated_link(x):
    return lower + (upper - lower) * func_link(x)

# Simple nonlinear probability (from adversarial Riesz paper)
def truncated_adv(x):
    return lower + (upper - lower) * (x > 0)

# Simple nonlinear probability (from adversarial Riesz paper)
def double_nonlinear(x):
    return lower + (upper - lower) * ((x < - 1/2) + (x < 1/2))

In [5]:
treatment_probability_func = truncated_logistic

#### True values: simulate ATE for experimental dataset

In [6]:
torch.manual_seed(123);

# Parameters
N = 500000
tmax = 1

# Higher dimensions = more sparsity. Minimum is dimX1 = dimX2 = 3
dimX = 3
dimS = 3

prob_G = 0.5 * torch.ones(N * tmax, 1)
G_all = torch.bernoulli(prob_G).reshape(-1,1)

X_all = torch.randn(N * tmax, dimX)

prob_D = X_all @ torch.tensor([-1, 1, -1] + [0] * (dimX - 3), dtype=torch.float32).reshape(-1,1) 
# prob_D = 0.5 * torch.ones(N * tmax, 1)
D_all = torch.bernoulli(treatment_probability_func(prob_D)).reshape(-1,1)

S0_all = X_all + torch.randn(N * tmax, dimS) * 0.5
S1_all = S0_all + 1 * 1 * (1 + torch.randn(N * tmax, dimS))
# S_all = (1 - G_all) * D_all * S1_all + (1 - (1 - G_all) * D_all) * S0_all


Y_eps = torch.randn(N * tmax, 1) * 1
# Y0_all = (S0_all @ torch.tensor([-1,1, -1] + [0] * (dimX - 3), dtype=torch.float32).reshape(-1,1) 
#       + X_all @ torch.tensor([1, 1, -1] + [0] * (dimX - 3), dtype=torch.float32).reshape(-1,1)
#       + Y_eps)

# Y1_all = (S1_all @ torch.tensor([1,1, -1] + [0] * (dimX - 3), dtype=torch.float32).reshape(-1,1) 
#       + X_all @ torch.tensor([1, 1, 1] + [0] * (dimX - 3), dtype=torch.float32).reshape(-1,1)
#       + Y_eps)

Y0_all =  (S0_all @ torch.tensor([1, 1, -1] + [0] * (dimX - 3), dtype=torch.float32).reshape(-1,1) 
      + X_all @ torch.tensor([1, 1, 1] + [0] * (dimX - 3), dtype=torch.float32).reshape(-1,1)
      + Y_eps)

Y1_all =  (S1_all @ torch.tensor([1,1, -1] + [0] * (dimX - 3), dtype=torch.float32).reshape(-1,1) 
      + X_all @ torch.tensor([1, 1, 1] + [0] * (dimX - 3), dtype=torch.float32).reshape(-1,1)
      + Y_eps)


In [7]:
true_value = torch.mean(Y1_all[((1 - G_all)).squeeze().bool(),:]) - torch.mean(Y0_all[((1 - G_all)).squeeze().bool(),:])
true_value

tensor(1.0057)

#### Simulation settings:

In [8]:
torch.manual_seed(123);

# Parameters
N = 500
tmax = 1
 
# Higher dimensions = more sparsity. Minimum is dimX1 = dimX2 = 3
dimX = 3
dimS = 3

#### Generate data

In [9]:
prob_G = 0.5 * torch.ones(N * tmax, 1)
G_all = torch.bernoulli(prob_G).reshape(-1,1)

X_all = torch.randn(N * tmax, dimX)

prob_D = X_all @ torch.tensor([-1, 1, -1] + [0] * (dimX - 3), dtype=torch.float32).reshape(-1,1) 
D_all = torch.bernoulli(treatment_probability_func(prob_D)).reshape(-1,1)

S0_all = X_all + torch.randn(N * tmax, dimS) * 0.5
S1_all = S0_all + 1 * 1 * (1 + torch.randn(N * tmax, dimS))
S_all = (1 - G_all) * D_all * S1_all + (1 - (1 - G_all) * D_all) * S0_all


Y_eps = torch.randn(N * tmax, 1) * 1

Y_all = G_all * (S_all @ torch.tensor([1,1, -1] + [0] * (dimX - 3), dtype=torch.float32).reshape(-1,1) 
      + X_all @ torch.tensor([1, 1, 1] + [0] * (dimX - 3), dtype=torch.float32).reshape(-1,1)
      + Y_eps)


In [10]:
print(G_all.shape)
print(D_all.shape)
print(X_all.shape)
print(Y_all.shape)
print(S_all.shape)

torch.Size([500, 1])
torch.Size([500, 1])
torch.Size([500, 3])
torch.Size([500, 1])
torch.Size([500, 3])


## Estimation:

#### Estimation settings

In [11]:
folds = 4
seed = 42
time0 = time.time()

#### Estimation 

In [23]:
pred_theta = torch.zeros(tmax,5)
pred_theta_changed= torch.zeros(tmax,5)
pred_sig = torch.zeros(tmax,5)
pred_sig_changed= torch.zeros(tmax,5)
for t in range(0,tmax):

    # Get data for current iteration
    G, X = G_all[(t) * N : (t+1) * N, :], X_all[(t) * N : (t+1) * N, :]
    D, S = D_all[(t) * N : (t+1) * N], S_all[(t) * N : (t+1) * N]
    Y = Y_all[(t) * N : (t+1) * N]
    D_changed = D.clone()
    Y_changed = Y.clone()
    ###################
    #D_changed[G.bool()] = D_changed[G.bool()]  # Crahses
    Y_changed[(1-G).bool()] =  0
   
    


    pred_theta[t,2], pred_sig[t,2] = estimateDynamicRiesz(Y, G, X, D, S, folds,
                                                                     method_a = "RF", lasso_a_settings = lasso_a_settings,
                                                                        method_f = "RF", lasso_f_settings = lasso_f_settings, seed= seed)
    pred_theta_changed[t,2], pred_sig_changed[t,2] = estimateDynamicRiesz(Y_changed, G, X, D_changed, S, folds,
                                                                     method_a = "RF", lasso_a_settings = lasso_a_settings,
                                                                        method_f = "RF", lasso_f_settings = lasso_f_settings, seed= seed)
                                                                     



print(pred_theta)
print(pred_theta_changed)

tensor([[0.0000, 0.0000, 1.1709, 0.0000, 0.0000]])
tensor([[0.0000, 0.0000, 1.1798, 0.0000, 0.0000]])


In [12]:
pred_theta = torch.zeros(tmax,5)
pred_sig = torch.zeros(tmax,5)

for t in range(0,tmax):

    # Get data for current iteration
    G, X = G_all[(t) * N : (t+1) * N, :], X_all[(t) * N : (t+1) * N, :]
    D, S = D_all[(t) * N : (t+1) * N], S_all[(t) * N : (t+1) * N]
    Y = Y_all[(t) * N : (t+1) * N]
    


    #---------------------------------------------------------------------------
    # # Estimator 1: Oracle
    # pi1, pi2_0, pi2_1  = pi1_all[(t) * N : (t+1) * N], pi2_0_all[(t) * N : (t+1) * N], pi2_1_all[(t) * N : (t+1) * N]
    # mu2_1, mu2_0, mu1_1, mu1_0 = mu2_1_all[(t) * N : (t+1) * N], mu2_0_all[(t) * N : (t+1) * N], mu1_1_all[(t) * N : (t+1) * N], mu1_0_all[(t) * N : (t+1) * N]
    # if d[0,0] == 1:
    #     RR2[:,t,:1], RR1[:,t,:1] = A1 * A2 / (pi1 * pi2_1), A1 / pi1 
    #     theta_hat = (RR2[:,t,:1] * Y - (RR1[:,t,:1] - 1) * mu1_1   - (RR2[:,t,:1] - RR1[:,t,:1]) * mu2_1)
    #     pred_theta[t,0] = torch.mean(theta_hat)
    #     pred_sig[t,0] = torch.sqrt(torch.mean((theta_hat - torch.mean(theta_hat) ) ** 2))
    # elif d[0,0] == 0:
    #     RR2[:,t,:1], RR1[:,t,:1] = (1 - A1) * (1 - A2) / ((1 - pi1  ) * (1 - pi2_0  )), (1 - A1) / (1 - pi1)
    #     theta_hat = (RR2[:,t,:1] * Y - (RR1[:,t,:1] - 1) * mu1_0   - (RR2[:,t,:1] - RR1[:,t,:1]) * mu2_0)
    #     pred_theta[t,0] = torch.mean(theta_hat)
    #     pred_sig[t,0] = torch.sqrt(torch.mean((theta_hat - torch.mean(theta_hat) ) ** 2))

    #---------------------------------------------------------------------------
    # Estimator 3: LASSO 

    pred_theta[t,2], pred_sig[t,2] = estimateDynamicRiesz(Y, G, X, D, S, folds,
                                                                     method_a = "LASSO", lasso_a_settings = lasso_a_settings,
                                                                        method_f = "LASSO", lasso_f_settings = lasso_f_settings, seed= seed)



    if t % 10 == 0:
        print("Time until iteration ",t, ": ", time.time() - time0)

print("Finished. Total time: ", time.time() - time0)


Time until iteration  0 :  1.203643798828125
Finished. Total time:  1.203773021697998


## Compute statistics

#### Get true value

#### Compute statistics

In [ ]:
bias = torch.mean(pred_theta[:t+1,:] - true_value, 0)
rmse = torch.sqrt(torch.mean( (pred_theta[:t+1,:] - true_value) ** 2, 0))

ub = pred_theta[:t+1,:] + 1.96 * pred_sig[:t+1,:] / torch.sqrt(torch.tensor(N))
lb = pred_theta[:t+1,:] - 1.96 * pred_sig[:t+1,:] / torch.sqrt(torch.tensor(N))

coverage = torch.mean( ( (true_value >= lb) & (true_value <= ub) ).float() , 0 )
interval_length = torch.mean(ub - lb,0)

# Create a DataFrame
data = {
    "Method": ["RF+LASSO", "LASSO+RF", "LASSO", "RF", "Net"],
    "Bias": bias.ravel(),
    "RMSE": rmse.ravel(),
    "Coverage": coverage.ravel(),
    "Interval Length": interval_length.ravel()
}

#### Print results

In [ ]:
df = pd.DataFrame(data)
print(df)

     Method      Bias      RMSE  Coverage  Interval Length
0  RF+LASSO -1.005658  1.005658       0.0         0.000000
1  LASSO+RF -1.005658  1.005658       0.0         0.000000
2     LASSO  0.227696  0.227696       1.0         1.088641
3        RF -1.005658  1.005658       0.0         0.000000
4       Net -1.005658  1.005658       0.0         0.000000


In [ ]:
pred_theta

tensor([[0.0000, 0.0000, 1.2334, 0.0000, 0.0000]])

In [ ]:
pred_theta

tensor([[ 0.0000,  0.0000, -0.0232,  0.0000,  0.0000]])

In [ ]:
# check_t = 0
# Compare RR:
# pd.DataFrame(torch.hstack((RR1[:,check_t,0:1], RR1[:,check_t,2:3], RR1[:,check_t,3:4], RR1[:,check_t,4:5]))).head(50)
# pd.DataFrame(torch.hstack((RR2[:,check_t,0:1], RR2[:,check_t,2:3], RR2[:,check_t,3:4], RR2[:,check_t,4:5]))).head(50)


In [ ]:
pred_theta

tensor([[0.0000, 0.0000, 1.2334, 0.0000, 0.0000]])